# 02 — Pipelines & ColumnTransformer

Chain preprocessing + modeling into one object to prevent data leakage and simplify deployment.

## Basic Pipeline

In [ ]:
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression()),
])

pipe.fit(X_train, y_train)
pipe.predict(X_test)
pipe.score(X_test, y_test)

# shorthand, auto-names steps from class names (lowercased)
pipe2 = make_pipeline(StandardScaler(), LogisticRegression())


## ColumnTransformer: different preprocessing per column type

In [ ]:
from sklearn.compose import ColumnTransformer, make_column_transformer, make_column_selector
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

numeric_features = ["age", "income", "credit_score"]
categorical_features = ["city", "employment_type"]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])

# select columns automatically by dtype instead of hardcoding names
auto_preprocessor = make_column_transformer(
    (StandardScaler(), make_column_selector(dtype_include="number")),
    (OneHotEncoder(handle_unknown="ignore"), make_column_selector(dtype_include="object")),
)


## Full Pipeline: preprocessing + model

In [ ]:
from sklearn.ensemble import RandomForestClassifier

full_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42)),
])

full_pipe.fit(X_train_df, y_train)
full_pipe.predict(X_test_df)


## Custom Transformers

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class LogTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, columns=None):
        self.columns = columns

    def fit(self, X, y=None):
        return self   # no fitting needed for this transform

    def transform(self, X):
        X = X.copy()
        cols = self.columns or X.columns
        X[cols] = np.log1p(X[cols])
        return X

pipe_with_custom = Pipeline([
    ("log", LogTransformer(columns=["income"])),
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression()),
])


## FeatureUnion: combine features from parallel pipelines

In [ ]:
from sklearn.pipeline import FeatureUnion
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest

combined_features = FeatureUnion([
    ("pca", PCA(n_components=5)),
    ("select_best", SelectKBest(k=3)),
])
combined_features.fit_transform(X_train, y_train)   # concatenates both outputs


## Accessing Steps & Feature Names

In [ ]:
full_pipe.named_steps["classifier"]           # access a step by name
full_pipe["preprocessor"]                     # or by indexing

full_pipe.named_steps["preprocessor"].get_feature_names_out()   # names after transformation


## Grid Search Over a Pipeline

Use `<step_name>__<param>` syntax to tune parameters at any stage.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "classifier__n_estimators": [100, 200],
    "classifier__max_depth": [None, 10, 20],
    "preprocessor__num__imputer__strategy": ["mean", "median"],
}

grid = GridSearchCV(full_pipe, param_grid, cv=5, scoring="accuracy", n_jobs=-1)
grid.fit(X_train_df, y_train)
grid.best_params_


## Common Gotchas
- Wrapping everything in a `Pipeline` is the main defense against data leakage — `cross_val_score`/`GridSearchCV` will correctly refit preprocessing per fold.
- `ColumnTransformer` drops columns not explicitly listed by default (`remainder="drop"`); pass `remainder="passthrough"` to keep them.
- Custom transformers must implement `fit`/`transform` (and inherit `TransformerMixin` for free `fit_transform`) to be pipeline-compatible.
- Parameter names for nested grid search use double underscores (`step__param`), and multiple levels of nesting chain further (`step__substep__param`).